# Static Background Subtraction

So far we looked at movement and location detection based on *adaptive* background subtraction, which is a useful approach  when dealing with a static camera but potentially slowly changing background (e.g., changing weather, lighting). Adaptive subtractors continuously update the reference frame, based on recent farmes, to prevent false detections. E.g., in a lab situation, a slowly changing backround may happen because of unintended lighting changes, or an animal changing the position of an object.

However, if you are confident that your background really is static, and the only thing that moves is your animal, you can also consider static background subtraction, which is simpler and allows for faster tracking. The idea is to create one static reference frame, and subtract that from each frame in the video. What remains should be the animal.

At the moment this notebook shows the principle. A next version of Birdwatcher will include a higher-level function that implements static background subtraction.

In [ ]:
import numpy as np
from itertools import repeat
import birdwatcher as bw
import birdwatcher.movementdetection as md
from birdwatcher.plotting import imshow_frame
from birdwatcher.utils import roi_to_npindex
import matplotlib.pyplot as plt
from pathlib import Path

We'll use the short zebra finch example video included in Birdwatcher.:

In [ ]:
vfs = bw.testvideostreamsmall()

Have a look at the video (press 'q' to quit the video before the end):

In [ ]:
vfs.show()

## Creating a reference frame

We didn't think of filming the background setting in this video without the bird, which would have been handy as a reference (average the frames of a few seconds of video of the empty cage). Now, we only have a video in which all frames contain the bird.

We'll create a reference by averaging 100 frames of video in one episode in the video where the birds sits on one perch, then set the values in a small rectangular mask over the bird to 'nan'. We'll do the same for a second episode when the bird sits at a different perch. The masks should not overlap. Averaging the two episodes, each with a nan-mask over the bird, will yield an empty cage.

In the beginning of the video, the bird sits on perch 3 (numbered from left to right). Let's define a rectangular region-of-interest, roi, that is not too large but that completely contains the bird:

In [ ]:
roi1 = (340, 450, 440, 560)
imshow_frame(vfs.get_frame(0), draw_rectangle=roi1)

This seems fine, but it is just for one frame. Ideally, we want our first reference to be based on the averrage of more frames to average out noise. Let's check if in the first 100 frames the bird indeed does not move beyond our defined `roi1`:

In [ ]:
vfs.iter_frames(startframe=0, nframes=100).draw_rectangles(repeat(roi1)).show()

It does not. Let's similarly look for an second episode in the video where the bird is on a different perch. Frame 270 seems a good candidate:

In [ ]:
roi2 = (340, 450, 690, 810)
imshow_frame(vfs.get_frame(270), draw_rectangle=roi2)

Check if bird does not leave the rectangle in the next 100 frames:

In [ ]:
vfs.iter_frames(startframe=170, nframes=100).draw_rectangles(repeat(roi2)).show()

It does not.

Next we calculate the average frame for both episodes:

In [ ]:
f1mean = vfs.iter_frames(startframe=0, nframes=100).calc_meanframe()
f2mean = vfs.iter_frames(startframe=170, nframes=100).calc_meanframe()

In [ ]:
imshow_frame(f1mean)

In [ ]:
imshow_frame(f2mean)

Video frames are based on numerical values of type 'uint8'. To set our regions of interest to 'nan', they need to be floating point values.

In [ ]:
f1mean = f1mean.astype('float64')
f2mean = f2mean.astype('float64')

Set the regions of interest to nan:

In [ ]:
f1mean[roi_to_npindex(roi1)] = np.nan
f2mean[roi_to_npindex(roi2)] = np.nan

Average the two with numpy's `nanmean`, function, round the results, and cast back to 'uint8', so that the results can be used as a video frame: 

In [ ]:
fmean = np.round(np.nanmean([f1mean,f2mean], axis=0)).astype('uint8')

In [ ]:
imshow_frame(fmean)

Voilla, an empty cage!

## Subtract reference frame from video

We can subtract the reference from our video by using the `absdiff_frame` method on a `Frames` object. Let's do that and look at the result (press 'q' to quit video output before end):

In [ ]:
vfs.iter_frames().absdiff_frame(fmean).show()

This looks promising! We should tweak this result a bit further though. We still have color data, and for quantitative movement and location analyses, we probably just want binary 'positive' pixels, indicating where the bird is. We also see that the shadow of the bird on the floor is also picked up. We should transform the color data to gray, apply some thresholding, which hopefully removes shadow and other lower-value noisy data, and make the outcome binary:

In [ ]:
(vfs.iter_frames()
    .absdiff_frame(fmean)
    .togray()
    .threshold(50, threshtype='binary')
    .draw_framenumbers()
    .show()
)

Nice! One could further apply methods to this result to make it more 'smooth', recover just the bird 'blob' etc. That is something for a different notebook. But, quickly trying out a `morphologyex` transform can't hurt:

In [ ]:
(vfs.iter_frames()
    .absdiff_frame(fmean)
    .togray()
    .threshold(50, threshtype='binary')
    .morphologyex(morphtype='close', kernelsize=5)
    .draw_framenumbers()
    .show()
)

## Saving results to Coordinate Array for later analyses of movement and position

We use the `save_nonzero` method to save the coordinates of non-zero (i.e. positive) pixels to a `CoordinateArrays` object:

In [ ]:
outputpath = Path('output')
if not outputpath.exists():
    outputpath.mkdir()

ca = (vfs.iter_frames()
    .absdiff_frame(fmean)
    .togray()
    .threshold(50, threshtype='binary')
    .morphologyex(morphtype='close', kernelsize=5)
    .save_nonzero(outputpath/'myresults', None)
)

sanity check a frame

In [ ]:
imshow_frame(ca.get_frame(200))

calculate mean of positive pixels for each frame

## Have a quick look at location

We can define bird location as the spatial mean of the blob of positive pixels:

In [ ]:
cm = ca.get_coordmean()

Plot the mean coordinates:

In [ ]:
plt.plot(cm)
plt.xlabel('frame number')
plt.ylabel('coordinate mean value')
plt.legend(['x', 'y'])

## Save results as a video in webm format for documentation

In [ ]:
filepath = outputpath / 'finch.webm'
codec = 'libvpx-vp9'
vformat = 'webm'
pixfmt = 'yuv420p'
(vfs.iter_frames()
    .absdiff_frame(fmean)
    .togray()
    .threshold(50, threshtype='binary')
    .resizebyfactor(0.7,0.7) # make it a bit smaller
    .morphologyex(morphtype='close', kernelsize=5)
    .draw_framenumbers()
    .tovideo(filepath, framerate=vfs.avgframerate, crf=None, codec=codec, pixfmt=pixfmt, vformat=vformat, overwrite=True)
)